# 08 — LangGraph: Agentes con estado y grafos de flujo

> **Bloque:** LLMs | **Nivel:** Avanzado
>
> Complementario al tutorial [08-langgraph.md](../../llms/08-langgraph.md)

Construimos un agente LangGraph paso a paso: desde un grafo mínimo hasta un sistema multi-agente con memoria persistente.

In [ ]:
# %pip install langgraph langchain-anthropic langchain-core

from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
import math

LLM = ChatAnthropic(model='claude-sonnet-4-6')
print('✓ Listo')

## 1. Grafo mínimo — un solo nodo

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def nodo_llm(state: State) -> dict:
    return {'messages': [LLM.invoke(state['messages'])]}

grafo = StateGraph(State)
grafo.add_node('llm', nodo_llm)
grafo.set_entry_point('llm')
grafo.add_edge('llm', END)
app_simple = grafo.compile()

# Visualización del grafo (Mermaid)
print(app_simple.get_graph().draw_mermaid())

# Primera prueba
res = app_simple.invoke({'messages': [HumanMessage(content='¿Qué es LangGraph en dos frases?')]})
print('\nRespuesta:', res['messages'][-1].content)

## 2. Edge condicional — ciclo agéntico

In [ ]:
# Herramientas
@tool
def calcular(expresion: str) -> float:
    """Evalúa una expresión matemática."""
    return eval(expresion, {'__builtins__': {}}, {'math': math})

@tool
def buscar_capital(pais: str) -> str:
    """Devuelve la capital de un país."""
    capitales = {'España': 'Madrid', 'Francia': 'París', 'Alemania': 'Berlín',
                 'Italia': 'Roma', 'Japón': 'Tokio', 'Portugal': 'Lisboa'}
    return capitales.get(pais, f"Capital de '{pais}' no encontrada")

@tool
def convertir_temp(valor: float, de: str, a: str) -> str:
    """Convierte temperatura entre Celsius, Fahrenheit y Kelvin."""
    if de == 'C' and a == 'F': r = valor * 9/5 + 32
    elif de == 'F' and a == 'C': r = (valor - 32) * 5/9
    elif de == 'C' and a == 'K': r = valor + 273.15
    elif de == 'K' and a == 'C': r = valor - 273.15
    else: return f'Conversión {de}→{a} no soportada'
    return f'{valor}°{de} = {r:.2f}°{a}'

herramientas = [calcular, buscar_capital, convertir_temp]
llm_tools = LLM.bind_tools(herramientas)

def nodo_llm_tools(state: State) -> dict:
    return {'messages': [llm_tools.invoke(state['messages'])]}

nodo_tools = ToolNode(herramientas)

# Grafo con ciclo
grafo2 = StateGraph(State)
grafo2.add_node('llm', nodo_llm_tools)
grafo2.add_node('herramientas', nodo_tools)
grafo2.set_entry_point('llm')
grafo2.add_conditional_edges('llm', tools_condition)  # → 'tools' o END
grafo2.add_edge('herramientas', 'llm')                # ciclo
app_tools = grafo2.compile()

print(app_tools.get_graph().draw_mermaid())

In [ ]:
# Prueba 1: cálculo
res = app_tools.invoke({'messages': [
    HumanMessage(content='¿Cuánto es math.pi elevado a 3? Da el resultado con 4 decimales.')
]})
print('→', res['messages'][-1].content)

In [ ]:
# Prueba 2: múltiples herramientas en una sola consulta
res = app_tools.invoke({'messages': [
    HumanMessage(content='¿Cuál es la capital de Japón? ¿Y cuánto es 212°F en Celsius?')
]})
print('→', res['messages'][-1].content)

## 3. Streaming — ver el grafo en acción paso a paso

In [ ]:
pregunta = '¿Cuánto es la raíz cuadrada de 1024? ¿Y la capital de Alemania?'
print(f'Pregunta: {pregunta}\n' + '─' * 50)

for evento in app_tools.stream({'messages': [HumanMessage(content=pregunta)]}):
    for nodo, valor in evento.items():
        print(f'\n[{nodo}]')
        if 'messages' in valor:
            ultimo = valor['messages'][-1]
            if hasattr(ultimo, 'content') and ultimo.content:
                print(f'  Texto: {ultimo.content[:300]}')
            if hasattr(ultimo, 'tool_calls') and ultimo.tool_calls:
                for tc in ultimo.tool_calls:
                    print(f'  🔧 {tc["name"]}({tc["args"]})')
            if hasattr(ultimo, 'tool_call_id'):
                print(f'  📦 Resultado: {ultimo.content}')

## 4. Memoria entre sesiones — MemorySaver

In [ ]:
checkpointer = MemorySaver()
app_memoria = grafo2.compile(checkpointer=checkpointer)

# Dos usuarios distintos con sus propios threads
config_ana   = {'configurable': {'thread_id': 'ana_001'}}
config_luis  = {'configurable': {'thread_id': 'luis_001'}}

# Ana pregunta
app_memoria.invoke({'messages': [HumanMessage(content='Me llamo Ana y soy matemática.')]}, config=config_ana)
res = app_memoria.invoke({'messages': [HumanMessage(content='¿Cómo me llamo y a qué me dedico?')]}, config=config_ana)
print('Ana:', res['messages'][-1].content)

# Luis en su propio thread (no conoce a Ana)
app_memoria.invoke({'messages': [HumanMessage(content='Me llamo Luis y soy arquitecto.')]}, config=config_luis)
res = app_memoria.invoke({'messages': [HumanMessage(content='¿Cuál es mi profesión?')]}, config=config_luis)
print('Luis:', res['messages'][-1].content)

In [ ]:
# Inspeccionar el estado guardado del thread de Ana
snapshot = app_memoria.get_state(config_ana)
print('Mensajes en el thread de Ana:')
for m in snapshot.values['messages']:
    rol = m.__class__.__name__.replace('Message', '')
    print(f'  [{rol}]: {m.content[:100]}')

## 5. Human-in-the-loop — pausa para aprobación

In [ ]:
class EstadoAprobacion(TypedDict):
    messages: Annotated[list, add_messages]
    plan: str

def generar_plan(state: EstadoAprobacion) -> dict:
    res = LLM.invoke(
        [SystemMessage(content='Genera un plan de acción detallado en 3 pasos para la tarea dada.')] +
        state['messages']
    )
    return {'messages': [res], 'plan': res.content}

def ejecutar_plan(state: EstadoAprobacion) -> dict:
    return {'messages': [AIMessage(content=f'✅ Plan ejecutado:\n{state["plan"]}')]}

cp = MemorySaver()
g_aprobacion = StateGraph(EstadoAprobacion)
g_aprobacion.add_node('planificar', generar_plan)
g_aprobacion.add_node('ejecutar', ejecutar_plan)
g_aprobacion.set_entry_point('planificar')
g_aprobacion.add_edge('planificar', 'ejecutar')
g_aprobacion.add_edge('ejecutar', END)

# interrupt_before='ejecutar' pausa el grafo ANTES de ejecutar
app_aprobacion = g_aprobacion.compile(checkpointer=cp, interrupt_before=['ejecutar'])

cfg = {'configurable': {'thread_id': 'tarea_001'}}

# Paso 1: generar plan (se pausa antes de ejecutar)
app_aprobacion.invoke(
    {'messages': [HumanMessage(content='Lanzar campaña de email marketing para el nuevo producto.')],
     'plan': ''},
    config=cfg
)
snapshot = app_aprobacion.get_state(cfg)
print('Plan propuesto:')
print(snapshot.values['plan'])
print(f'\nPróximo paso (pausado): {snapshot.next}')

In [ ]:
# Paso 2: el humano aprueba → reanudar desde donde se pausó
aprobado = True  # simula la decisión humana

if aprobado:
    resultado = app_aprobacion.invoke(None, config=cfg)  # None = reanudar
    print(resultado['messages'][-1].content)
else:
    print('Plan rechazado. El grafo no se reanuda.')

## 6. Sistema multi-agente — Supervisor + subagentes

In [ ]:
class EstadoMulti(TypedDict):
    messages: Annotated[list, add_messages]
    siguiente: str
    pasos: int

ROLES = ['investigador', 'redactor', 'revisor']

def supervisor(state: EstadoMulti) -> dict:
    prompt = f"""Coordinas un equipo: {', '.join(ROLES)}.
Basándote en el estado de la tarea, elige quién actúa ahora.
Responde SOLO con el nombre del agente o 'FINISH' si la tarea está lista."""
    res = LLM.invoke([SystemMessage(content=prompt)] + state['messages'])
    sig = res.content.strip()
    if sig not in ROLES + ['FINISH']:
        sig = 'FINISH'
    return {'siguiente': sig, 'pasos': state['pasos'] + 1}

def crear_agente(rol: str, instruccion: str):
    def nodo(state: EstadoMulti) -> dict:
        res = LLM.invoke([SystemMessage(content=instruccion)] + state['messages'])
        return {'messages': [AIMessage(content=f'[{rol.upper()}] {res.content}')]}
    nodo.__name__ = rol
    return nodo

def enrutar(state: EstadoMulti) -> str:
    if state['siguiente'] == 'FINISH' or state['pasos'] >= 5:
        return END
    return state['siguiente']

gm = StateGraph(EstadoMulti)
gm.add_node('supervisor', supervisor)
gm.add_node('investigador', crear_agente('investigador',
    'Eres un investigador. Analiza el tema y extrae los puntos clave con datos relevantes.'))
gm.add_node('redactor', crear_agente('redactor',
    'Eres un redactor. Usa la investigación para escribir contenido claro y estructurado.'))
gm.add_node('revisor', crear_agente('revisor',
    'Eres un revisor. Corrige y mejora el contenido redactado. Sé específico en tus sugerencias.'))

gm.set_entry_point('supervisor')
gm.add_conditional_edges('supervisor', enrutar,
    {'investigador': 'investigador', 'redactor': 'redactor', 'revisor': 'revisor', END: END})
for rol in ROLES:
    gm.add_edge(rol, 'supervisor')

app_multi = gm.compile()

print(app_multi.get_graph().draw_mermaid())

In [ ]:
resultado = app_multi.invoke({
    'messages': [HumanMessage(content='Escribe un párrafo explicando las ventajas de los modelos de IA locales.')],
    'siguiente': '',
    'pasos': 0
})

print('\n=== RESULTADO FINAL ===')
for m in resultado['messages']:
    if isinstance(m, AIMessage):
        print(f'\n{m.content[:400]}')

## 7. Tu turno — experimenta

Prueba a modificar alguno de los grafos anteriores:

- Añade una herramienta nueva al agente del paso 2 (por ejemplo, una que busque en Wikipedia)
- Cambia el `interrupt_before` para pausar antes de un nodo diferente
- Añade un cuarto subagente al sistema multi-agente (p. ej. un `formateador`)

In [ ]:
# Espacio libre para experimentar

@tool
def mi_herramienta(parametro: str) -> str:
    """Describe aquí tu herramienta."""
    return f'Resultado de mi_herramienta con: {parametro}'

# Tu código aquí...

---
**Tutorial relacionado:** [08-langgraph.md](../../llms/08-langgraph.md) · **Siguiente:** [09-modelos-locales-ollama.ipynb](./09-modelos-locales-ollama.ipynb)